# Notebook 01 - Data Understanding

## Problem contract

Predict `Will_Buy_EV` for each `id` in the test set.

- Task: binary classification on tabular data.
- Raw target labels are the strings `No` and `Yes`; they are validated and then
  encoded explicitly as `No -> 0`, `Yes -> 1`.
- Submission stores a probability, not a hard label.
- Primary metric: ROC-AUC. Guardrails: PR-AUC and LogLoss.
- Validation: 5-fold x 3-repeat stratified cross-validation from a persisted
  split manifest.

Every numeric claim below is produced by the cell directly above it.

## Setup and path resolution

Resolve the project root from the notebook location so the notebook runs from a
clean kernel without absolute paths.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ev_purchase.config import load_config
from ev_purchase.data import DatasetContract, encode_target, load_frames, validate_frames

SEED = 42
np.random.seed(SEED)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

config = load_config(PROJECT_ROOT / "configs" / "base.yaml", project_root=PROJECT_ROOT)
print("project root resolved:", PROJECT_ROOT.name)
print("train:", config.data.train.relative_to(PROJECT_ROOT))
print("test:", config.data.test.relative_to(PROJECT_ROOT))

## Load data and assert the schema contract

`validate_frames` enforces the dataset contract: exact columns, unique ids, no
train/test id overlap, and raw target labels restricted to `No`/`Yes`.

In [ ]:
train_df, test_df = load_frames(config.data.train, config.data.test)
contract = DatasetContract(
    id_column=config.data.id_column,
    target_column=config.data.target_column,
)
validate_frames(train_df, test_df, contract)

print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
print("raw target labels:", sorted(train_df[config.data.target_column].unique()))
print("schema contract: OK")

### Observation / Interpretation / Action

- **Observation:** the contract validator accepted both frames and the raw
  target contains exactly the two labels printed above.
- **Interpretation:** the data matches the agreed contract, so no silent label
  remapping is needed or permitted.
- **Action:** encode the target explicitly with `encode_target` and keep the
  string labels out of every downstream model input.

## Feature-role map

Split the columns into identifier, target, numeric predictors, and categorical
predictors. This map drives preprocessing decisions later.

In [ ]:
id_col = config.data.id_column
target_col = config.data.target_column

predictors = train_df.drop(columns=[id_col, target_col])
numeric_cols = predictors.select_dtypes(include="number").columns.tolist()
categorical_cols = [c for c in predictors.columns if c not in numeric_cols]

role_map = pd.DataFrame(
    [{"column": c, "role": "numeric"} for c in numeric_cols]
    + [{"column": c, "role": "categorical"} for c in categorical_cols]
)
print("identifier:", id_col, "| target:", target_col)
print("n_predictors:", predictors.shape[1])
role_map

### Observation / Interpretation / Action

- **Observation:** the predictor block splits into the numeric and categorical
  columns listed above.
- **Interpretation:** both families are present, so the baseline needs scaling
  plus encoding while gradient-boosted trees can consume categories natively.
- **Action:** keep two model paths - a linear baseline behind a preprocessor and
  GBDTs with native categorical handling.

## Data quality audit

Check missing values, duplicate ids, duplicate rows, constant columns, and
quasi-constant columns before trusting any model result.

In [ ]:
audit = {
    "train_nulls": int(train_df.isna().sum().sum()),
    "test_nulls": int(test_df.isna().sum().sum()),
    "duplicate_ids_train": int(train_df[id_col].duplicated().sum()),
    "duplicate_ids_test": int(test_df[id_col].duplicated().sum()),
    "id_overlap_train_test": len(set(train_df[id_col]) & set(test_df[id_col])),
    "duplicate_rows_excluding_id": int(predictors.duplicated().sum()),
    "constant_columns": [c for c in predictors.columns if predictors[c].nunique() <= 1],
}
for key, value in audit.items():
    print(f"{key}: {value}")

dominance = (
    predictors.apply(lambda s: s.value_counts(normalize=True).iloc[0])
    .sort_values(ascending=False)
    .rename("top_value_share")
    .to_frame()
)
dominance

### Observation / Interpretation / Action

- **Observation:** the audit counters above are all zero and no column is
  constant; the highest single-value share is the first row of the table.
- **Interpretation:** there is no missing-value imputation problem, no duplicate
  leakage between train and test, and no degenerate column to drop.
- **Action:** skip imputation-driven feature engineering, but keep the imputers
  in the pipeline as a defensive contract for future data drift.

## Target prevalence

Measure class balance, because it decides whether accuracy is meaningless and
whether PR-AUC deserves guardrail status.

In [ ]:
y_encoded = encode_target(train_df[target_col], contract)
counts = pd.Series(y_encoded).value_counts().sort_index()
positive_rate = float(y_encoded.mean())

print("label counts (0=No, 1=Yes):")
print(counts.to_string())
print(f"positive rate: {positive_rate:.4%}")
print(f"majority-class accuracy floor: {1 - positive_rate:.4%}")

### Observation / Interpretation / Action

- **Observation:** the positive class is the minority at the rate printed above,
  so a do-nothing classifier already reaches the accuracy floor shown.
- **Interpretation:** accuracy is not a usable objective here; ranking quality
  and probability calibration matter instead.
- **Action:** keep ROC-AUC as the primary metric and report PR-AUC and LogLoss
  as guardrails, exactly as the design specifies.

## Numeric diagnostics

Summarise the numeric predictors and their univariate association with the
encoded target.

In [ ]:
numeric_summary = predictors[numeric_cols].describe().T
numeric_summary["corr_with_target"] = [
    float(np.corrcoef(predictors[c], y_encoded)[0, 1]) for c in numeric_cols
]
numeric_summary.sort_values("corr_with_target", ascending=False)

### Observation / Interpretation / Action

- **Observation:** the numeric columns sit on very different scales and their
  point-biserial correlations with the target are listed above.
- **Interpretation:** the scale spread hurts a linear model but not trees, and
  no single numeric column looks like a target proxy.
- **Action:** standardise numerics for the linear baseline only, and rely on the
  cross-validated comparison rather than these univariate signals.

## Categorical diagnostics

Report cardinality and per-category positive rate to expose both encoding cost
and signal strength.

In [ ]:
rows = []
for col in categorical_cols:
    grouped = pd.DataFrame({col: predictors[col], "y": y_encoded}).groupby(col)["y"]
    rates = grouped.mean()
    rows.append(
        {
            "column": col,
            "cardinality": int(predictors[col].nunique()),
            "min_positive_rate": float(rates.min()),
            "max_positive_rate": float(rates.max()),
            "spread": float(rates.max() - rates.min()),
        }
    )

categorical_summary = pd.DataFrame(rows).sort_values("spread", ascending=False)
categorical_summary

### Observation / Interpretation / Action

- **Observation:** every categorical column has low cardinality and the
  positive-rate spread per column is shown above.
- **Interpretation:** one-hot encoding is cheap at this cardinality, and the
  columns with the widest spread carry the most univariate signal.
- **Action:** use plain one-hot encoding for the linear baseline; no target
  encoding is required, which removes a common leakage source.

## Train/test distribution shift

Compare the predictor distributions across train and test. A large shift would
invalidate cross-validation as a proxy for leaderboard performance.

In [ ]:
shift_rows = []
for col in numeric_cols:
    shift_rows.append(
        {
            "column": col,
            "train_mean": float(train_df[col].mean()),
            "test_mean": float(test_df[col].mean()),
            "abs_mean_diff": abs(float(train_df[col].mean()) - float(test_df[col].mean())),
        }
    )
for col in categorical_cols:
    tr_share = train_df[col].value_counts(normalize=True)
    te_share = test_df[col].value_counts(normalize=True)
    aligned = pd.concat([tr_share, te_share], axis=1).fillna(0.0)
    shift_rows.append(
        {
            "column": col,
            "train_mean": float("nan"),
            "test_mean": float("nan"),
            "abs_mean_diff": float((aligned.iloc[:, 0] - aligned.iloc[:, 1]).abs().max()),
        }
    )

pd.DataFrame(shift_rows).sort_values("abs_mean_diff", ascending=False)

### Observation / Interpretation / Action

- **Observation:** the univariate train/test difference summary is sorted above
  by maximum absolute mean discrepancy for numerics and category proportions.
- **Interpretation:** univariate summaries provide screening only and cannot
  detect joint or multivariate distribution shifts between train and test.
- **Action:** multivariate adversarial validation below determines whether
  detectable shift warrants investigation before finalizing the validation strategy.

## Adversarial validation

Evaluate multivariate train-vs-test domain shift using a 5-fold StratifiedKFold LightGBM classifier.
The domain label is `0` for train and `1` for test. Identifier and true target columns are strictly
excluded to eliminate trivial leakage. Preprocessing (handling categoricals) is fitted inside each
training fold only, preventing cross-fold leakage.

In [ ]:
from ev_purchase.adversarial_validation import (
    compute_file_sha256,
    run_adversarial_validation,
)

# Strict exclusion assertion: drop id and target from predictors
_train_features = train_df.drop(columns=[config.data.id_column, config.data.target_column])
assert config.data.id_column not in _train_features.columns
assert config.data.target_column not in _train_features.columns

# Compute real dataset SHA256 hashes to guarantee provenance in persisted artifacts
train_sha = compute_file_sha256(config.data.train)
test_sha = compute_file_sha256(config.data.test)
data_sha256 = {
    "train_sha256": train_sha,
    "test_sha256": test_sha,
}

artifacts_dir = PROJECT_ROOT / "outputs" / "diagnostics"
adversarial_res = run_adversarial_validation(
    train_df=train_df,
    test_df=test_df,
    id_column=config.data.id_column,
    target_column=config.data.target_column,
    n_splits=5,
    random_state=SEED,
    artifacts_dir=artifacts_dir,
    data_sha256=data_sha256,
)

oof_domain_preds = adversarial_res.oof_domain_preds
top_features = adversarial_res.feature_importance.head(10)

print("adversarial validation metrics:")
for k, v in adversarial_res.metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.6f}")
    else:
        print(f"  {k}: {v}")

print()
print("top 10 shift features (fold-averaged importance):")
print(top_features.to_string(index=False))

print()
print("=== Adversarial Validation Diagnostic Findings ===")
print(f"shift band: {adversarial_res.interpretation['shift_band']}")
print(f"summary:    {adversarial_res.interpretation['summary']}")
print(f"action:     {adversarial_res.interpretation['action']}")
print(f"caveats:    {adversarial_res.interpretation['caveats']}")

print()
print("persisted diagnostic artifacts:")
print(f"  adversarial_validation.json: {(artifacts_dir / 'adversarial_validation.json').exists()}")
print(f"  adversarial_fold_metrics.csv: {(artifacts_dir / 'adversarial_fold_metrics.csv').exists()}")
print(f"  adversarial_feature_importance.csv: {(artifacts_dir / 'adversarial_feature_importance.csv').exists()}")

### Observation / Interpretation / Action

- **Observation:** inspect the 5-fold cross-validated domain ROC-AUC, PR-AUC, LogLoss, and fold-averaged feature importances printed above from `adversarial_res`.
- **Interpretation Bands (Diagnostic Heuristic):**
  - **ROC-AUC <= 0.55 (Little detectable shift):** domain classifier cannot meaningfully distinguish train from test predictors; covariate shift is minimal under this model family.
  - **0.55 < ROC-AUC <= 0.60 (Mild shift):** subtle distributional divergence in one or more features; warrants monitoring leaderboard alignment.
  - **ROC-AUC > 0.60 (Material diagnostic shift):** domain origin is reliably predictable from features; local validation splits may systematically diverge from test evaluation without adjustment.
- **Action Guidance:** follow the dynamically recommended action printed above for the current run's observed shift band before freezing the validation strategy.

## Leakage audit

Confirm that nothing in the predictor block trivially reveals the target and
that the identifier carries no signal.

In [ ]:
id_corr = float(np.corrcoef(train_df[id_col], y_encoded)[0, 1])
perfect_separators = [
    col
    for col in predictors.columns
    if pd.DataFrame({"x": predictors[col], "y": y_encoded}).groupby("x")["y"].nunique().eq(1).all()
]

print(f"id vs target correlation: {id_corr:.6f}")
print("columns that perfectly separate the target:", perfect_separators)
print("target column present in predictors:", target_col in predictors.columns)
print("id column present in predictors:", id_col in predictors.columns)

### Observation / Interpretation / Action

- **Observation:** the identifier correlation is effectively zero, no predictor
  perfectly separates the target, and neither `id` nor the target leaks into the
  predictor block.
- **Interpretation:** there is no obvious target leakage; remaining leakage risk
  comes from how we fit preprocessing, not from the raw columns.
- **Action:** fit every transformer inside the training fold only, which the
  training engine already enforces.

## Repeated cross-validation rationale

Load the persisted split manifest and verify it is the single source of truth
for fold assignment.

In [ ]:
manifest_path = PROJECT_ROOT / "outputs" / "splits" / "repeated_stratified_5x3.parquet"
manifest = pd.read_parquet(manifest_path)

per_row = manifest.groupby("row_index").size()
print("manifest rows:", len(manifest))
print("distinct (repeat, fold) groups:", manifest.groupby(["repeat", "fold"]).ngroups)
print("validation assignments per row - min:", int(per_row.min()), "max:", int(per_row.max()))
print("rows covered:", per_row.shape[0], "of", len(train_df))

### Observation / Interpretation / Action

- **Observation:** the manifest holds the row count printed above across 15
  `(repeat, fold)` groups, and every training row receives the same number of
  validation assignments.
- **Interpretation:** repeating the 5-fold split three times averages away
  fold-assignment noise, which matters when comparing close candidates.
- **Action:** score every experiment on this manifest so runs stay comparable,
  and reject any run whose split hash differs.

## Decision ledger

Decisions carried into Notebook 02. Each one is justified by an executed cell
above, not by assumption.

In [ ]:
decisions = pd.DataFrame(
    [
        {
            "decision": "Validate raw No/Yes then encode No->0, Yes->1",
            "evidence": "schema contract cell",
        },
        {
            "decision": "Primary ROC-AUC with PR-AUC and LogLoss guardrails",
            "evidence": "target prevalence cell",
        },
        {"decision": "Skip imputation-driven features", "evidence": "data quality audit cell"},
        {"decision": "One-hot categoricals for the baseline", "evidence": "categorical cell"},
        {"decision": "No target encoding", "evidence": "categorical cardinality cell"},
        {"decision": "Trust CV over leaderboard feedback", "evidence": "train/test shift cell"},
        {"decision": "Fit preprocessing inside the fold only", "evidence": "leakage audit cell"},
        {"decision": "Score on the persisted 5x3 manifest", "evidence": "manifest cell"},
    ]
)
decisions

## Deferred experiments

Ideas deliberately postponed so Notebook 02 stays a bounded, finite matrix.

In [ ]:
deferred = pd.DataFrame(
    [
        {"idea": "Numeric interaction features", "reason": "No baseline gap justifies it yet"},
        {"idea": "Target/count encoding", "reason": "Cardinality is low; adds leakage risk"},
        {"idea": "Extensive HPO", "reason": "Runs in Notebook 04 after baselines settle"},
        {"idea": "Neural tabular models", "reason": "Design fixes a GBDT-first scope"},
        {"idea": "Pseudo-labelling the test set", "reason": "Unverifiable without a holdout"},
    ]
)
deferred

### Observation / Interpretation / Action

- **Observation:** the ledger and deferral tables above are both populated.
- **Interpretation:** the data is clean, balanced-enough to rank with ROC-AUC,
  and free of obvious leakage, so modelling can start immediately.
- **Action:** hand off to Notebook 02 to register baselines and define the
  experiment matrix.